# Tutorial -  Mean Reversion
Sergio Cabrales, Universidad de los Andes

https://www.sac2.com/

## 1. Carga de librerías, funciones y APIs necesarias.

#### 1.2. Se cargan las librerías requeridas

In [ ]:
# Funciones numéricas adicionales
import numpy as np

# Lectura de datos y manejo de Data-sets
import pandas as pd

# Datos
import yfinance as yfin

# Gráficos
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
import seaborn as sns

#analisis tecnico
import mplfinance as mpf

# Probabilidad y estadística
import math
from scipy.stats import norm, chi2, jarque_bera,shapiro
from scipy.optimize import brentq
from scipy import stats

In [ ]:
# FRED
from fredapi import Fred

In [ ]:
# FRED_API_KEY
fred = Fred(api_key='a3d8a45391e92cc5c30c9c255819c138')

## 2. Obtención de datos históricos

#### 2.1. Descarga de datos desde la FRED

https://fred.stlouisfed.org/


In [ ]:
# Descargamos datos FRED
df = pd.DataFrame(fred.get_series('POILBREUSDM','2005-01-01'))
df = df.dropna()
#df.rename(columns={"0": "Adj Close"}, inplace=True)
df.index.names = ['Date']
df.columns = ["Close"]
df

In [ ]:
df.to_excel('brent_oil_prices.xlsx', index=True)

## 3. Visualización y Estadísticas Descriptivas

### 3.1. Grafica de precios

In [ ]:
#Plot adj close
plt.figure(figsize=(15,8))
plt.plot(df['Close'], color = 'blue')
plt.title('Precios')
plt.xlabel('Fecha')
plt.ylabel('Precio ($)')
plt.show()

## 4. Retornos

### 4.1. Retornos Logarítmicos

Los retornos logarítmicos se calculan como:
$$
r_{t} = ln \left( \frac{S_t}{S_{t-1}} \right ) = ln \left( S_{t} \right) - ln \left( S_{t-1} \right)
$$

In [ ]:
# Guardamos los retornos logaritmicos en una nueva columna.
df['Log_Returns'] = np.log(df['Close']) - np.log(df['Close'].shift(1))
df['Log_Returns'][0] = 0
df

### 4.3. Retornos Logarítmicos anualizados

Podemos calcular el log-retorno anual ($r$) como el número de días bursátiles (252 días) por el promedio del log-retorno diario:

$$
r = 252 \bar{r_t}
$$

In [ ]:
# Podemos imprimir el retornos anual:
LogReturns = np.mean(df['Log_Returns'])*12
LogReturns

### 4.4. Gráfica de retornos
- Podemos graficar los retornos igual que como graficamos los precios.

In [ ]:
# Gráfico de los retornos logarítmicos
plt.figure(figsize=(15,8))
plt.plot(df['Log_Returns'], color = 'red')
plt.title('Retornos Logarítmicos')
plt.xlabel('Fecha')
plt.show()

## 5. Volatilidad

### 5.1 Volatilidad diaria y anual

La volatilidad diaria del activo es la desviación estándar de sus retornos o la raíz de la varianza:  

$$vol=desv(r)=\sqrt{Var(r)}$$

En finanzas, se utiliza con mayor frecuencia la volatilidad anualizada ($\sigma$) en lugar de la volatilidad diaria. Teniendo en cuenta que en cada año hay 252 días bursátiles:

$$ \sigma^{2} = \sum_{1}^{252} Var_{diaria}$$
$$ \sigma^{2} = 252 \sigma_{diaria}^{2}$$

Se saca la raíz cuadra a ambos lados para calcular la volatilidad:

$$ \sqrt{\sigma^{2}} = \sqrt{252 \sigma_{diaria}^{2}}$$
$$ \sigma = \sigma_{diaria} \sqrt{252}$$

In [ ]:
# Calculamos la volatilidad diaria con los retornos logaritmicos.
vol_d = np.std(df['Log_Returns'])

# Anualizamos la volatilidad diaria.
sigma = vol_d * np.sqrt(12)

print("Volatilidad diaria: {:.4f} %".format(100*vol_d))
print("Volatilidad anualizada: {:.4f} %".format(100*sigma))

In [ ]:
!pip install arch
from arch import arch_model

# Preparar los retornos (multiplicamos por 100 para mejor convergencia del modelo)
returns = df['Log_Returns'].dropna() * 100

# Especificar y ajustar el modelo GARCH(1,1)
model_garch = arch_model(returns, vol='Garch', p=1, q=1, rescale=False)
resultado_garch = model_garch.fit(disp='off')

# Extraer la volatilidad condicional (último valor de la serie)
# Se divide por 100 para regresar a la escala original
vol_mensual_garch = resultado_garch.conditional_volatility.iloc[-1] / 100

# Anualizar la volatilidad (multiplicando por raíz de 12 ya que los datos son mensuales)
sigma_anual_garch = vol_mensual_garch * np.sqrt(12)

print(resultado_garch.summary())
print("\n=======================================================")
print(f"Volatilidad mensual condicional (GARCH): {vol_mensual_garch*100:.4f}%")
print(f"Sigma anualizado (GARCH): {sigma_anual_garch*100:.4f}%")
sigma = sigma_anual_garch

## Augmented Dickey–Fuller test (ADF)

In [ ]:
from statsmodels.tsa.stattools import adfuller

In [ ]:
# test for stationarity using the ADF test
result = adfuller(df['Close'], autolag='AIC')
print("ADF statistic: ", result[0])
print("p-value: ", result[1])
print("Critical values: ", result[4])

# print the results
if result[1] < 0.05:
    print("The series is stationary")
else:
    print("The series is not stationary")

## Mean Reversion Model

In [ ]:
y = df['Close'] - df['Close'].shift(1)
y

In [ ]:
X = sm.add_constant(df['Close'].shift(1))
X

In [ ]:
# Define the mean reversion model
model = sm.OLS(y, X, missing='drop')

# Fit the model
result = model.fit()

# Print the summary of the regression results
print(result.summary())

In [ ]:
a = result.params[0]
a

In [ ]:
b = result.params[1]
b

In [ ]:
#obtain standardized residuals
residuals = result.resid
sigma_e = np.std(result.resid)
sigma_e

In [ ]:
# Long-term mean
m = -a/b
m

In [ ]:
# Speed of mean reversion
eta = -np.log(1+b)
eta

In [ ]:
# Parameters
m = m      # Long-term mean
sigma = sigma     # Volatility
eta = eta   # Speed of mean reversion
N = 12
dt = 1/N    # Time step
T = 1         # Time horizon
n = 10000      # Number of simulations
S0 = df['Close'][-1]       # Initial price
S0

In [ ]:
# Function to simulate paths
def simulate_path(m, sigma, eta, dt, T, n, S0, sigma_e):
    timesteps = int(T/dt)
    paths = np.zeros((n, timesteps+1))
    paths[:,0] = S0

    for i in range(n):
        for j in range(1, timesteps+1):
          paths[i,j] = m*(1-np.exp(-eta))+ np.exp(-eta)*paths[i,j-1] + np.random.normal(loc=0,scale=sigma_e)

    return paths

In [ ]:
# Simulate paths
paths = simulate_path(m, sigma, eta, dt, T, n, S0, sigma_e)

In [ ]:
# Plot paths
timesteps = int(T/dt)
t = np.linspace(0, T, timesteps+1)
plt.figure(figsize=(10,6))
for i in range(n):
    plt.plot(t, paths[i])
plt.xlabel('Time (years)')
plt.ylabel('Price (USD)')
plt.title('Simulated Brent Oil Prices')
plt.show()

In [ ]:
last_values = paths[:, -1]

percentile_5 = np.percentile(last_values, 5)
percentile_50 = np.percentile(last_values, 50)
percentile_95 = np.percentile(last_values, 95)

print(f"Percentil 5: {percentile_5:.2f}")
print(f"Percentil 50 (Mediana): {percentile_50:.2f}")
print(f"Percentil 95: {percentile_95:.2f}")